In [1]:
from pathlib import Path

# The notebooks are stored in <package>/notebooks.
notebooks_dir = Path.cwd()
if notebooks_dir.name != "notebooks":
    candidate = notebooks_dir / "notebooks"
    if candidate.exists():
        notebooks_dir = candidate

project_root = notebooks_dir.parent
nhanes_dir = project_root / "src" / "nhanes"
jordan_dir = project_root / "src" / "jordan_external_validation"
expected_nhanes = project_root / "outputs" / "nhanes"
expected_jordan = project_root / "outputs" / "jordan"

assert (project_root / "README.md").exists(), project_root
print("Project root:", project_root)

Project root: <project_root>


In [2]:
import json
import numpy as np
import pandas as pd

nhanes_live = nhanes_dir / "outputs"
nhanes_results = nhanes_live if (nhanes_live / "common_logistic_results.csv").exists() else expected_nhanes

jordan_live = jordan_dir / "external_validation" / "jordan_results"
jordan_results = jordan_live if (jordan_live / "jordan_external_validation_headline_results.csv").exists() else expected_jordan

In [3]:
nhanes_models = pd.read_csv(nhanes_results / "common_logistic_results.csv")
nhanes_pairs = pd.read_csv(nhanes_results / "paired_logistic_auc_differences.csv")
jordan_models = pd.read_csv(jordan_results / "jordan_external_validation_headline_results.csv")
jordan_pairs = pd.read_csv(jordan_results / "jordan_external_validation_auc_contrasts.csv")

primary = nhanes_models.query("panel == 'mec_nonfasting' and config == 'BOTH'").iloc[0]
assert round(primary.auc, 4) == 0.8863
assert round(primary.brier, 4) == 0.0866

nhanes_whr_bmi = nhanes_pairs.query("panel == 'mec_nonfasting' and comparison == 'WHR minus BMI'").iloc[0]
assert round(nhanes_whr_bmi.delta_auc, 4) == 0.0063
assert round(nhanes_whr_bmi.lower_psu, 4) == -0.0034
assert round(nhanes_whr_bmi.upper_psu, 4) == 0.0149

jordan_bmi = jordan_models.query("configuration == 'BMI'").iloc[0]
jordan_whr = jordan_models.query("configuration == 'WHR'").iloc[0]
assert round(jordan_bmi.weighted_auc, 4) == 0.7617
assert round(jordan_whr.weighted_auc, 4) == 0.7378

jordan_contrast = jordan_pairs.query("contrast == 'BMI minus WHR'").iloc[0]
assert round(jordan_contrast.auc_difference, 4) == 0.0239
assert round(jordan_contrast.lower_psu, 4) == -0.0018
assert round(jordan_contrast.upper_psu, 4) == 0.0559

print("PASS: principal NHANES and Jordan values match the reported results.")

PASS: principal NHANES and Jordan values match the reported results.


In [4]:
frozen_dir = jordan_dir / "external_validation" / "frozen_nhanes_models"
reduced_split = pd.read_csv(frozen_dir / "nhanes_cohort_and_split.csv").set_index("split")
reduced = pd.read_csv(frozen_dir / "nhanes_internal_test_results.csv").set_index("config")
nhanes_missing = expected_nhanes / "mec_nonfasting_missingness.csv"
if (nhanes_live / "mec_nonfasting_missingness.csv").exists():
    nhanes_missing = nhanes_live / "mec_nonfasting_missingness.csv"
missingness = pd.read_csv(nhanes_missing).set_index("variable")


def value_range(series):
    return round(series.min(), 3), round(series.max(), 3)


# Reduced-model development cohort (NHANES men aged 18-69)
assert reduced_split.loc["all", "n"] == 2120
assert [reduced_split.loc[s, "n"] for s in ["train", "validation", "test"]] == [1283, 410, 427]
assert reduced_split.loc["test", "cases"] == 60
assert round(reduced_split.loc["test", "weighted_case_fraction"] * 100, 1) == 12.3

# Internal reduced-model performance (Results and Supplementary Table S68)
expected_auc = {"BASE": 0.8525, "BMI": 0.8504, "WHR": 0.8583, "BOTH": 0.8546}
for config, auc in expected_auc.items():
    assert round(reduced.loc[config, "auc"], 4) == auc, config
assert round(reduced.loc["BMI", "auc"] - reduced.loc["WHR", "auc"], 4) == -0.0079
assert value_range(reduced.sensitivity) == (0.633, 0.648)
assert value_range(reduced.specificity) == (0.898, 0.935)

expected_limits = {"BASE": (0.7353, 0.9492), "BMI": (0.7434, 0.9413), "WHR": (0.7565, 0.9363), "BOTH": (0.7651, 0.9323)}
for config, (lower, upper) in expected_limits.items():
    limits = pd.read_csv(frozen_dir / f"{config}_internal_test_psu_intervals.csv").set_index("metric").loc["auc"]
    assert (round(limits.lower_psu, 4), round(limits.upper_psu, 4)) == (lower, upper), config

# External sensitivity and specificity at the frozen 0.20 threshold (Limitations, Reviewer 2.8)
assert value_range(jordan_models.sensitivity_at_0_20) == (0.256, 0.359)
assert value_range(jordan_models.specificity_at_0_20) == (0.898, 0.921)

# Primary panel: both anthropometric measures added to the BASE predictors
both_base = nhanes_pairs.query("panel == 'mec_nonfasting' and comparison == 'BOTH minus BASE'").iloc[0]
assert (round(both_base.delta_auc, 4), round(both_base.lower_psu, 4), round(both_base.upper_psu, 4)) == (-0.0021, -0.0112, 0.0041)

# Missing anthropometry imputed, not excluded (Methods, Reviewer 2.3)
assert missingness.loc["bmi", "missing_n"] == 47
assert missingness.loc["waist_to_hip_ratio", "missing_n"] == 138

print("PASS: reduced-model, threshold, BOTH-minus-BASE and missingness values match the manuscript.")

PASS: reduced-model, threshold, BOTH-minus-BASE and missingness values match the manuscript.


In [5]:
print("Saved missing-value counts in the primary panel:")
print(missingness.loc[["bmi", "waist_to_hip_ratio"]].round(1).to_string())
assert round(missingness.loc["bmi", "missing_percent"], 1) == 1.8
assert round(missingness.loc["waist_to_hip_ratio", "missing_percent"], 1) == 5.3

raw_dir = project_root / "data" / "raw"
if (raw_dir / "DEMO_J.xpt").exists():
    demo = pd.read_sas(raw_dir / "DEMO_J.xpt", format="xport")
    bmx = pd.read_sas(raw_dir / "BMX_J.xpt", format="xport")
    diq = pd.read_sas(raw_dir / "DIQ_J.xpt", format="xport")
    merged = demo.merge(bmx, on="SEQN", how="left").merge(diq, on="SEQN", how="left")
    men = merged[(merged.RIAGENDR == 1) & (merged.RIDAGEYR >= 18) & merged.DIQ010.isin([1, 2]) & (merged.WTMEC2YR > 1)]
    whr = men.BMXWAIST / men.BMXHIP
    print()
    print("Recount from raw NHANES files:")
    print("Men:", len(men))
    print("Missing BMI from raw data:", int(men.BMXBMI.isna().sum()), f"({men.BMXBMI.isna().mean() * 100:.1f}%)")
    print("Missing WHR from raw data:", int(whr.isna().sum()), f"({whr.isna().mean() * 100:.1f}%)")
    assert len(men) == 2589
    assert int(men.BMXBMI.isna().sum()) == 47
    assert int(whr.isna().sum()) == 138
    print("PASS: raw-data recount matches the saved missing-value counts.")
else:
    print()
    print("Raw NHANES files not found; recount skipped. Run nhanes/download_sources.py to enable it.")

Saved missing-value counts in the primary panel:
                    missing_n  missing_percent
variable                                      
bmi                        47              1.8
waist_to_hip_ratio        138              5.3

Recount from raw NHANES files:
Men: 2589
Missing BMI from raw data: 47 (1.8%)
Missing WHR from raw data: 138 (5.3%)
PASS: raw-data recount matches the saved missing-value counts.


In [6]:
cohort_counts = json.loads((expected_nhanes / "cohort_outcome_counts.json").read_text())
print("NHANES cohort counts:", {k: v for k, v in cohort_counts.items() if k != "source"})
expected_counts = {"merged_n": 9254, "adult_men_n": 2840, "definite_outcome_n": 2753, "positive_reports": 477, "borderline_n": 86, "unknown_n": 1}
for key, value in expected_counts.items():
    assert cohort_counts[key] == value, key

if (raw_dir / "DIQ_J.xpt").exists():
    adult_men = merged[(merged.RIAGENDR == 1) & (merged.RIDAGEYR >= 18)]
    print("Recount from raw DIQ010: borderline =", int((adult_men.DIQ010 == 3).sum()), "| unknown =", int(adult_men.DIQ010.isin([7, 9]).sum()))
    assert int((adult_men.DIQ010 == 3).sum()) == 86
    assert len(adult_men) == 2840

predictions_file = jordan_live / "jordan_external_validation_predictions.csv"
if predictions_file.exists():
    jordan_predictions = pd.read_csv(predictions_file)
    prevalence = np.average(jordan_predictions.diabetes, weights=jordan_predictions.analysis_weight)
    print("Jordan weighted prevalence recomputed from licensed predictions:", f"{prevalence * 100:.1f}%")
else:
    prevalence = json.loads((expected_jordan / "jordan_outcome_prevalence.json").read_text())["weighted_prevalence"]
    print("Jordan weighted prevalence from packaged aggregate:", f"{prevalence * 100:.1f}%")
assert round(prevalence * 100, 1) == 23.0

primary_table = nhanes_models.query("panel == 'mec_nonfasting'").set_index("config").loc[["BASE", "BMI", "WHR", "BOTH"]]
print()
print("Table 2 average precision (AP) and PR AUC:")
print(primary_table[["average_precision", "pr_auc"]].round(4).to_string())
expected_ap_pr = {"BASE": (0.5854, 0.5717), "BMI": (0.5732, 0.5617), "WHR": (0.6296, 0.6197), "BOTH": (0.6322, 0.6211)}
for config, (ap, pr) in expected_ap_pr.items():
    assert round(primary_table.loc[config, "average_precision"], 4) == ap, config
    assert round(primary_table.loc[config, "pr_auc"], 4) == pr, config

print("PASS: cohort audit counts, Jordan prevalence and Table 2 AP/PR AUC match the manuscript.")

NHANES cohort counts: {'merged_n': 9254, 'adult_men_n': 2840, 'definite_outcome_n': 2753, 'positive_reports': 477, 'borderline_n': 86, 'unknown_n': 1}
Recount from raw DIQ010: borderline = 86 | unknown = 1
Jordan weighted prevalence from packaged aggregate: 23.0%

Table 2 average precision (AP) and PR AUC:
        average_precision  pr_auc
config                           
BASE               0.5854  0.5717
BMI                0.5732  0.5617
WHR                0.6296  0.6197
BOTH               0.6322  0.6211
PASS: cohort audit counts, Jordan prevalence and Table 2 AP/PR AUC match the manuscript.
